# FINS notebook 10 — an emulator audit with a known answer (exp68; Colab, GPU)

Registered in `README_EXPERIMENTS.md` (exp68, EP0–EP5, 10.10) before any of these runs.
* cell 2: twelve FNO2d emulators of pyqg at 128^2, seeds 42–44, in this order: f-plane **plain**, beta-plane **plain**
  (PG8 redone with a better recipe), f-plane **rot** (every training sample rotated by a random angle) and f-plane
  **grid** (coordinates as two extra input channels). The recipe fixes PG8's two limits: 40 Fourier modes instead of
  24 (24 cannot carry 0.3 % of the increment's energy) and 1024 training trajectories instead of 256 (a generalization
  gap). 10000 steps each, roughly 45–70 min on a T4 (the first log lines show the speed); each emulator saves
  `<checkpoint>.partial` to Drive every 5000 steps (about 0.3 GB; files Colab deletes or overwrites on Drive go to the
  Trash and still count against the quota, so empty the Trash when space runs short) and resumes from it: rerun cells 1
  and 2. Generating each seed's 1024 training trajectories takes about 5 min of CPU before its first training.
* cell 3: per emulator, the one-step error, the commutators ||F(Tq) - T F(q)|| / ||F(q) - q|| with rotations and shifts
  (the direct reference) and FINS on its own rollouts (exp65). A few minutes per emulator.
* cell 4 shows progress; cell 5 zips the json and logs to `fins_nb10.zip` (small: upload this one to the chat), in
  /content and on My Drive. The emulators (about 100 MB each) stay on Drive under fins_runs/results/pyqg.

Rerun after Drive ran out of space: cell 2 skips every emulator whose three audits exist (its checkpoint is not needed)
and starts over from an unreadable `.partial`; only f rot s44 remains (about 50 min with its audits).

If GPU time runs short: the six plain emulators (EP0–EP2) first, then rot (EP3–EP4), then grid (EP5). Cell 3 audits
whatever is trained; run cells 3 and 5 at any point and send the zip.

Before running, upload the newest **`fins_repo_full.zip`** from the chat to My Drive (root or one folder down).

In [ ]:
# 1. setup: Drive, the newest repository zip (with exp68), pyqg from GitHub, GPU check
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.environ['TQDM_DISABLE'] = '1'
R = os.environ['FINS_RESULTS_DIR']
os.makedirs(os.path.join(R, 'pyqg'), exist_ok=True)


def newest(pattern, test):
    cands = sorted(glob.glob(f'/content/drive/MyDrive/{pattern}') + glob.glob(f'/content/drive/MyDrive/*/{pattern}'),
                   key=os.path.getmtime, reverse=True)
    for z in cands:
        ok = test(z)
        print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  usable: {ok}')
        if ok:
            return z


def has_exp68(z):
    with zipfile.ZipFile(z) as zz:
        name = 'fins_repo/experiments/exp68_emulator_pairs.py'
        return name in zz.namelist() and b'def read_partial' in zz.read(name)


ZIP_PATH = newest('fins_repo_full*.zip', has_exp68)
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive has exp68 with read_partial: upload the newest zip from the chat')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
EXPDIR = '/content/fins/fins_repo/experiments'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/pyqg/pyqg.git'], check=False)
for mod in ('exp68_emulator_pairs', 'exp65_emulator_audit'):
    chk = subprocess.run([sys.executable, '-c', f'import {mod}'], cwd=EXPDIR, capture_output=True, text=True)
    if chk.returncode:
        raise SystemExit(f'{mod} does not import here:\n' + chk.stderr[-2000:])
chk = subprocess.run([sys.executable, 'exp68_emulator_pairs.py', 'check'], cwd=EXPDIR, capture_output=True, text=True)
print(chk.stdout[-1500:], chk.stderr[-1500:])
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU (or better), then rerun this cell')
CACHE = '/content/pairs_cache'
RUNS = os.path.join(R, 'pairs_runs')
os.makedirs(os.path.join(RUNS, 'logs'), exist_ok=True)
SEEDS = (42, 43, 44)
ROWS = (('f', 'plain'), ('beta', 'plain'), ('f', 'rot'), ('f', 'grid'))


def go(stem, cmd, keys):
    if os.path.exists(stem + '.json'):
        print('done', os.path.basename(stem))
        return
    log = os.path.join(RUNS, 'logs', os.path.basename(stem) + '.log')
    t0 = time.time()
    with open(log, 'a') as fh:
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        rc = subprocess.call([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
    lines = open(log).read().splitlines()
    keep = [l.strip() for l in lines if any(k in l for k in keys)] if rc == 0 else lines[-6:]
    print(os.path.basename(stem), 'exit', rc, f'{time.time() - t0:.0f} s', *[k[:160] for k in keep[-6:]], sep='\n   ')


def ckpt(regime, arm, s):
    return os.path.join(R, 'pyqg', f'fno2d_{regime}_q_n128_s{s}_ex68{arm}.pt')

In [ ]:
# 2. train the twelve emulators (resumable): f plain, beta plain, f rot, f grid
order = [(r, a, s) for r, a in ROWS for s in SEEDS]
def audited(regime, arm, s):
    return all(os.path.exists(os.path.join(RUNS, f'ex68_{w}_{regime}_{arm}_s{s}.json')) for w in ('error', 'commute', 'fins'))


for regime, arm, s in order:
    if os.path.exists(ckpt(regime, arm, s)) or audited(regime, arm, s):
        print('trained' if os.path.exists(ckpt(regime, arm, s)) else 'audited (checkpoint not needed)', regime, arm, s)
        continue
    log = os.path.join(RUNS, 'logs', f'train_{regime}_{arm}_s{s}.log')
    t0 = time.time()
    print(f'{regime} {arm} s{s}: training (log {log})', flush=True)
    with open(log, 'a') as fh:
        cmd = ['exp68_emulator_pairs.py', 'train', '--regime', regime, '--arm', arm, '--seed', str(s), '--cache', CACHE,
               '--ckpt-every', '5000']
        fh.write('$ ' + ' '.join(cmd) + '\n')
        fh.flush()
        p = subprocess.Popen([sys.executable, '-u', *cmd], cwd=EXPDIR, stdout=fh, stderr=subprocess.STDOUT)
        seen = len(open(log).read().splitlines())
        while p.poll() is None:
            time.sleep(60)
            lines = open(log).read().splitlines()
            for l in lines[seen:]:
                print('   ', l.strip()[:160], flush=True)
            seen = len(lines)
    lines = open(log).read().splitlines()
    print(f'{regime} {arm} s{s}: exit {p.returncode}, {time.time() - t0:.0f} s', *[l[:160] for l in lines[seen:]],
          sep='\n   ')
left = [f'{r} {a} s{s}' for r, a, s in order if not (os.path.exists(ckpt(r, a, s)) or audited(r, a, s))]
print('all twelve emulators trained' if not left else 'NOT YET: ' + ', '.join(left))

In [ ]:
# 3. audit each trained emulator: one-step error, rotation commutator, FINS on its own rollouts
for s in SEEDS:
    for regime, arm in ROWS:
        if not os.path.exists(ckpt(regime, arm, s)):
            print('not trained yet:', regime, arm, s)
            continue
        base = ['--regime', regime, '--arm', arm, '--seed', str(s)]
        for what, keys in (('error', ('one step',)), ('commute', ('commutator',))):
            stem = os.path.join(RUNS, f'ex68_{what}_{regime}_{arm}_s{s}')
            go(stem, ['exp68_emulator_pairs.py', what, *base, '--save', stem], keys)
        stem = os.path.join(RUNS, f'ex68_fins_{regime}_{arm}_s{s}')
        go(stem, ['exp65_emulator_audit.py', 'fins', '--regime', regime, '--seed', str(s), '--fno-tag', f'_ex68{arm}',
                  '--save', stem], ('quotients', 'rotation plane', 'own rollouts'))

In [ ]:
# 4. progress
nt = sum(os.path.exists(ckpt(r, a, s)) for r, a in ROWS for s in SEEDS)
na = len(glob.glob(os.path.join(RUNS, 'ex68_*.json')))
print(f'trained {nt}/12; audit results {na}/36')
for f in sorted(glob.glob(os.path.join(R, 'pyqg', '*_ex68*.partial'))):
    print('in progress:', os.path.basename(f), time.ctime(os.path.getmtime(f)))

In [ ]:
# 5. copy the json and logs file by file to local disk (remount if Drive drops); fins_nb10.zip in /content and on Drive
local = '/content/fins_nb10'
shutil.rmtree(local, ignore_errors=True)
files = glob.glob(os.path.join(RUNS, '*.json')) + glob.glob(os.path.join(RUNS, 'logs', '*.log'))
for f in sorted(set(files)):
    dst = os.path.join(local, os.path.relpath(f, R))
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    for attempt in range(3):
        try:
            shutil.copy2(f, dst)
            break
        except OSError as e:
            print('retry', f, e)
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)
out = '/content/fins_nb10.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(local, '**', '*'), recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, local))
print(out, os.path.getsize(out), 'bytes')
try:
    shutil.copy2(out, '/content/drive/MyDrive/fins_nb10.zip')
    print('copied to MyDrive: fins_nb10.zip (upload this one to the chat)')
except OSError as e:
    print('Drive copy failed; download /content/fins_nb10.zip from the file pane:', e)